# Zambia VACS 2014 — PUD exploration

Public-use microdata live under `data/raw/Zambia Stata/`. **Male** and **Female** respondent files are **separate** (split-sample EAs); structure is almost identical (`id` ranges do not overlap between files).

This notebook: load → raw samples → **harmonized slot summary** (table below) → **§2** variable table → **`utils.checklist`** (**`type_and_width`** + TSV) → further use of `df`.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ZAMBIA_DIR = ROOT / "data" / "raw" / "Zambia Stata"
MALE_DTA = ZAMBIA_DIR / "ZAMBIA_VACS_2014_Male_PUD.dta"
FEMALE_DTA = ZAMBIA_DIR / "ZAMBIA_VACS_2014_Female_PUD.dta"

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

## 1. Load data

Primary frame: **`ZAMBIA_VACS_2014_Male_PUD.dta`**. The female file uses the same variable names for design/geo fields; row count differs (891 females, 928 males).

In [2]:
DTA_PATH = MALE_DTA
if not DTA_PATH.is_file():
    raise FileNotFoundError(f"Expected:\n  {DTA_PATH}")

df, meta = pyreadstat.read_dta(DTA_PATH)
print(f"File: {DTA_PATH}")
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]:,}")
if FEMALE_DTA.is_file():
    df_f, _ = pyreadstat.read_dta(FEMALE_DTA)
    print(f"Female PUD (for reference): {df_f.shape[0]:,} × {df_f.shape[1]:,}")
if getattr(meta, "file_label", None):
    print(f"Stata dataset label: {meta.file_label!r}")
df.head()

File: /Users/starsrain/research_side_projects_ipv/data/raw/Zambia Stata/ZAMBIA_VACS_2014_Male_PUD.dta
Rows × columns: 928 × 1,093
Female PUD (for reference): 891 × 1,093


,id,Q402,Q402B,psu,hh,prov,dist,const,ward,csa,...,Q1401,Q1402,Q1403,Q1404,Q1405,Q1406,Q402I,Q402BI,PROV_CODE,Finalwgt
0,001009,1,NaN,1,9,1,101,1,2,1,...,2,2,2,,,NO COMMENTS,NaN,NaN,1,2010.990702
1,001020,1,NaN,1,20,1,101,1,2,1,...,2,2,2,,,NO COMMENT,NaN,NaN,1,1005.495351
2,001024,1,NaN,1,24,1,101,1,2,1,...,2,2,2,,,NO COMMENTS,NaN,NaN,1,2010.990702
3,001028,NaN,NaN,1,28,1,101,1,2,1,...,1,1,2,,,,NaN,NaN,1,3108.861056
4,001035,1,NaN,1,35,1,101,1,2,1,...,1,1,2,,,NO COMMENTS,NaN,NaN,1,3108.861056


In [6]:
df['Q2'].agg(['mean', 'median', 'std', 'min', 'max'])

mean      18.310345
median    18.000000
std        3.385453
min       13.000000
max       24.000000
Name: Q2, dtype: float64

### Raw row samples

Head / tail / fixed random sample, plus a **narrow** subset of IDs, geography, cluster, and interview date fields.

In [5]:
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("--- head(8) — all columns ---")
display(df.head(8))

print("\n--- tail(4) ---")
display(df.tail(4))

print("\n--- sample(6, random_state=0) ---")
display(df.sample(6, random_state=0))

_core_cols = [
    c
    for c in [
        "id",
        "psu",
        "hh",
        "prov",
        "dist",
        "const",
        "ward",
        "csa",
        "nsel",
        "ntot",
        "HYR_VF",
        "HMTH_VF",
        "HDAY_VF",
        "VISIT_NF",
        "hcluster",
        "Q2"
    ]
    if c in df.columns
]
subset = df[_core_cols]
print(f"\n--- head(8) / sample — {len(_core_cols)} columns ---")
display(subset.head(8))
display(subset.sample(6, random_state=0))

--- head(8) — all columns ---


,id,Q402,Q402B,psu,hh,prov,dist,const,ward,csa,INT_COD,hcluster,VISIT_NF,HYR_VF,HMTH_VF,...,Q1312,Q1313,Q1314,Q1315,Q1400,Q1401,Q1402,Q1403,Q1404,Q1405,Q1406,Q402I,Q402BI,PROV_CODE,Finalwgt
0,001009,1,NaN,1,9,1,101,1,2,1,1012,1,1,2014,9,...,NaN,1,2,2,1,2,2,2,,,NO COMMENTS,NaN,NaN,1,2010.990702
1,001020,1,NaN,1,20,1,101,1,2,1,1012,1,1,2014,9,...,1,1,2,2,1,2,2,2,,,NO COMMENT,NaN,NaN,1,1005.495351
2,001024,1,NaN,1,24,1,101,1,2,1,1011,1,1,2014,9,...,NaN,2,2,1,1,2,2,2,,,NO COMMENTS,NaN,NaN,1,2010.990702
3,001028,NaN,NaN,1,28,1,101,1,2,1,1011,1,1,2014,9,...,1,2,2,2,1,1,1,2,,,,NaN,NaN,1,3108.861056
4,001035,1,NaN,1,35,1,101,1,2,1,1011,1,1,2014,9,...,NaN,2,2,2,1,1,1,2,,,NO COMMENTS,NaN,NaN,1,3108.861056
5,001050,NaN,NaN,1,50,1,101,1,2,1,1014,1,1,2014,9,...,NaN,1,2,2,1,2,2,2,,,NO COMMENTS,NaN,NaN,1,1554.430528
6,001062,NaN,NaN,1,62,1,101,1,2,1,1014,1,1,2014,9,...,NaN,1,2,2,1,2,2,2,,,COMMENTS,NaN,NaN,1,1005.495351
7,001077,NaN,NaN,1,77,1,101,1,2,1,1013,1,1,2014,9,...,NaN,1,2,2,1,2,2,2,,,,NaN,NaN,1,1554.430528



--- tail(4) ---


,id,Q402,Q402B,psu,hh,prov,dist,const,ward,csa,INT_COD,hcluster,VISIT_NF,HYR_VF,HMTH_VF,...,Q1312,Q1313,Q1314,Q1315,Q1400,Q1401,Q1402,Q1403,Q1404,Q1405,Q1406,Q402I,Q402BI,PROV_CODE,Finalwgt
924,240024,1,NaN,240,24,10,1007,147,5,2,1101,1,2,2014,9,...,1,1,1,2,1,2,2,2,,,NO,NaN,NaN,10,1012.922925
925,241001,NaN,NaN,241,1,10,1007,147,12,6,1102,1,1,2014,9,...,NaN,2,2,2,2,2,2,2,,,NO COMENT,NaN,NaN,10,949.453124
926,241009,NaN,NaN,241,9,10,1007,147,12,6,1102,1,1,2014,9,...,NaN,2,2,2,2,2,2,2,,,NO COMENT.,NaN,NaN,10,949.453124
927,241021,NaN,NaN,241,21,10,1007,147,12,6,1101,1,2,2014,9,...,NaN,2,99,2,1,2,2,2,,,NO COMMENT.,NaN,NaN,10,876.745937



--- sample(6, random_state=0) ---


,id,Q402,Q402B,psu,hh,prov,dist,const,ward,csa,INT_COD,hcluster,VISIT_NF,HYR_VF,HMTH_VF,...,Q1312,Q1313,Q1314,Q1315,Q1400,Q1401,Q1402,Q1403,Q1404,Q1405,Q1406,Q402I,Q402BI,PROV_CODE,Finalwgt
308,070069,1,NaN,70,69,3,303,46,11,7,1035,1,1,2014,9,...,NaN,2,2,2,1,2,2,2,,,NONE,NaN,NaN,3,869.373631
352,076058,NaN,NaN,76,58,3,306,52,8,2,1035,1,1,2014,9,...,NaN,2,2,2,2,2,2,2,,,NONE,NaN,NaN,3,1246.136987
666,175080,1,NaN,175,80,7,705,95,8,5,1075,1,1,2014,9,...,NaN,2,2,2,1,2,2,2,,,NO COMMENTS,NaN,NaN,7,2772.265682
912,239015,1,NaN,239,15,10,1004,144,13,6,1103,1,1,2014,9,...,NaN,2,2,2,99,2,2,2,,,NO COMMENTS,NaN,NaN,10,840.795501
815,215036,NaN,NaN,215,36,9,908,127,3,4,1091,1,1,2014,9,...,NaN,1,2,2,99,2,2,2,,,NO COMMENT,NaN,NaN,9,4301.472650
231,042058,NaN,NaN,42,58,2,210,35,25,4,1023,1,1,2014,9,...,1,1,2,2,2,2,2,2,,,WHAT BENEFIT IS THERE IN GATHERING THE ANSWERS...,NaN,NaN,2,2680.256722



--- head(8) / sample — 16 columns ---


,id,psu,hh,prov,dist,const,ward,csa,nsel,ntot,HYR_VF,HMTH_VF,HDAY_VF,VISIT_NF,hcluster,Q2
0,001009,1,9,1,101,1,2,1,2,9,2014,9,13,1,1,18
1,001020,1,20,1,101,1,2,1,1,3,2014,9,13,1,1,23
2,001024,1,24,1,101,1,2,1,2,10,2014,9,13,1,1,21
3,001028,1,28,1,101,1,2,1,2,12,2014,9,13,1,1,15
4,001035,1,35,1,101,1,2,1,2,9,2014,9,12,1,1,15
5,001050,1,50,1,101,1,2,1,1,5,2014,9,13,1,1,17
6,001062,1,62,1,101,1,2,1,1,2,2014,9,13,1,1,18
7,001077,1,77,1,101,1,2,1,1,8,2014,9,13,1,1,17


,id,psu,hh,prov,dist,const,ward,csa,nsel,ntot,HYR_VF,HMTH_VF,HDAY_VF,VISIT_NF,hcluster,Q2
308,070069,70,69,3,303,46,11,7,1,2,2014,9,17,1,1,16
352,076058,76,58,3,306,52,8,2,1,2,2014,9,13,1,1,19
666,175080,175,80,7,705,95,8,5,2,9,2014,9,19,1,1,19
912,239015,239,15,10,1004,144,13,6,1,2,2014,9,24,1,1,20
815,215036,215,36,9,908,127,3,4,4,8,2014,9,18,1,1,21
231,042058,42,58,2,210,35,25,4,2,6,2014,9,6,1,1,19


### Harmonized codebook slots — Zambia 2014 (Male PUD; Female PUD parallel)

Use this table when filling your Excel draft. Every row includes **data source**, **variable name(s)**, and notes with **(a) data type** and **(b) digits / character width** (from the PUDs; verify analysis rules in `ZAMBIA_VACS_2014_DataUserGuide.pdf`).

**Shared data source:** `ZAMBIA_VACS_2014_Male_PUD.dta` and `ZAMBIA_VACS_2014_Female_PUD.dta` (same variables for design/geo/id fields; row counts differ).

| Slot | Data source | Variable(s) | Note |
|------|-------------|-------------|------|
| Respondent ID | Male + Female PUD above | `id` | **(a)** Type: `str` (characters are digits 0–9 only). **(b)** Width: 6 characters. Unique within each file; IDs do not overlap across Male vs Female PUD. |
| Household ID | Male + Female PUD above | `psu` + `hh` | **(a)** Type: `int` + `int` (EA ID + household number within EA). **(b)** Digits: `psu` is 1–3 in Male PUD, 2–3 in Female PUD; `hh` is 1–3 in both. Composite `psu`+`hh` is unique per row in each file. |
| Geo level 1 | Male + Female PUD above | `prov` | **(a)** Type: `int` (province code). **(b)** Digits: 1–2 (values 1–10 in both PUDs). |
| Geo level 2 (district code) | Male + Female PUD above | `dist` | **(a)** Type: `int` (district code). **(b)** Digits: 3–4 (Male ~101–1007; Female ~101–1006). **Not** the EA / Admin 2 slot for this project—see **`psu`**. |
| Geo level 2 / Admin 2 — EA | Male + Female PUD above | `psu` | **Enumeration area** (Stata *EA ID*). Same variable as **Cluster** below—cross-reference in Excel **notes** (`skills/memory.md`). |
| Cluster | Male + Female PUD above | `psu` | **(a)** Type: `int` (EA / PSU for `svy`). **(b)** Digits: same as `psu` in Household row. |
| Interview date | Male + Female PUD above | `HYR_VF` + `HMTH_VF` + `HDAY_VF` | **(a)** Type: three separate `int` fields (final visit year, month, day). **(b)** Digits: year 4; month 1–2 (e.g. 8–10 in extract); day 1–2 (e.g. 1–30 Male, 1–28 Female). |

In [4]:
L = meta.column_names_to_labels or {}

def slot_summary(name, cols, df):
    """One slot: single column or composite (join with '+'; uniqueness on concatenated key)."""
    missing = [c for c in cols if c not in df.columns]
    if missing:
        return {"slot": name, "variable": "; ".join(cols), "note": f"MISSING: {missing}"}
    if len(cols) == 1:
        col = cols[0]
        s = df[col]
        lbl = L.get(col, "") or ""
        if pd.api.types.is_numeric_dtype(s):
            width = f"values min={int(s.min())}, max={int(s.max())}"
        else:
            lens = s.astype(str).str.len()
            width = f"{int(lens.min())}-{int(lens.max())} chars"
        uni = s.nunique(dropna=True)
        return {
            "slot": name,
            "variable": col,
            "stata_label": (lbl or "")[:80],
            "dtype": str(s.dtype),
            "n_distinct": int(uni),
            "n_rows": len(s),
            "unique_per_row": bool(uni == len(s) and s.notna().all()),
            "width_hint": width,
            "sample": list(s.dropna().head(5)),
        }
    parts = [df[c].astype(str) for c in cols]
    key = parts[0]
    for p in parts[1:]:
        key = key + "_" + p
    uni = key.nunique()
    labels_join = " | ".join((L.get(c, "") or "")[:40] for c in cols)
    return {
        "slot": name,
        "variable": " + ".join(cols),
        "stata_label": labels_join[:200],
        "dtype": "composite (" + ", ".join(str(df[c].dtype) for c in cols) + ")",
        "n_distinct": int(uni),
        "n_rows": len(df),
        "unique_per_row": bool(uni == len(df) and not key.duplicated().any()),
        "width_hint": "per-component ranges below",
        "sample": df[cols].head(5).to_dict(orient="records"),
    }


PIPELINE = [
    ("1. Respondent ID", ["id"]),
    ("2. Household ID", ["psu", "hh"]),
    ("3. Geo level 1 (region/province)", ["prov"]),
    ("4. District code (not EA)", ["dist"]),
    ("5. Admin 2 / EA & cluster (psu)", ["psu"]),
    ("6. Interview date", ["HYR_VF", "HMTH_VF", "HDAY_VF"]),
]

print("Data source:", DTA_PATH.name)
rows = [slot_summary(name, cols, df) for name, cols in PIPELINE]
pipe_df = pd.DataFrame(rows)
display(pipe_df.drop(columns=["sample"], errors="ignore"))

print("\nSamples:")
for r in rows:
    print(r["slot"], "→", r.get("sample"))

print("\n--- Frequency: prov, dist (top 12) ---")
display(df["prov"].value_counts().head(12))
display(df["dist"].value_counts().head(12))

print("\n--- Cross-check: female file `id` overlap with male (expect 0) ---")
if FEMALE_DTA.is_file():
    df_f, _ = pyreadstat.read_dta(FEMALE_DTA)
    ov = set(df["id"].astype(str)) & set(df_f["id"].astype(str))
    print("overlap count:", len(ov))

Data source: ZAMBIA_VACS_2014_Male_PUD.dta


,slot,variable,stata_label,dtype,n_distinct,n_rows,unique_per_row,width_hint
0,1. Respondent ID,id,UNIQUE IDENTIFICATION,str,928,928,True,6-6 chars
1,2. Household ID,psu + hh,EA ID | household,"composite (int64, int64)",928,928,True,per-component ranges below
2,3. Geo level 1 (region/province),prov,province,int64,10,928,False,"values min=1, max=10"
3,4. Geo level 2 (district),dist,district,int64,61,928,False,"values min=101, max=1007"
4,5. Cluster (EA / PSU),psu,EA ID,int64,133,928,False,"values min=1, max=241"
5,6. Interview date,HYR_VF + HMTH_VF + HDAY_VF,Final Visit Date -Year | Final Visit Date -Mon...,"composite (int64, int64, int64)",46,928,False,per-component ranges below



Samples:
1. Respondent ID → ['001009', '001020', '001024', '001028', '001035']
2. Household ID → [{'psu': 1, 'hh': 9}, {'psu': 1, 'hh': 20}, {'psu': 1, 'hh': 24}, {'psu': 1, 'hh': 28}, {'psu': 1, 'hh': 35}]
3. Geo level 1 (region/province) → [1, 1, 1, 1, 1]
4. Geo level 2 (district) → [101, 101, 101, 101, 101]
5. Cluster (EA / PSU) → [1, 1, 1, 1, 1]
6. Interview date → [{'HYR_VF': 2014, 'HMTH_VF': 9, 'HDAY_VF': 13}, {'HYR_VF': 2014, 'HMTH_VF': 9, 'HDAY_VF': 13}, {'HYR_VF': 2014, 'HMTH_VF': 9, 'HDAY_VF': 13}, {'HYR_VF': 2014, 'HMTH_VF': 9, 'HDAY_VF': 13}, {'HYR_VF': 2014, 'HMTH_VF': 9, 'HDAY_VF': 12}]

--- Frequency: prov, dist (top 12) ---


prov
2     181
5     132
9     126
3     119
7      77
10     77
1      73
4      57
6      46
8      40
Name: count, dtype: int64

dist
504     107
210      46
204      44
302      33
403      32
205      25
101      24
303      23
704      23
1004     21
207      20
307      20
Name: count, dtype: int64


--- Cross-check: female file `id` overlap with male (expect 0) ---
overlap count: 0


## 2. What’s in this file?

Stata labels, dtypes, missingness (first 40 variables + highest missing %). Run the **checklist** cell next for **`type_and_width`**, **`suggested_layout`**, and TSV (`utils.checklist`); edit **`CANDIDATES`** as needed.

In [4]:
name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}
var_table = pd.DataFrame({
    "column": df.columns,
    "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (100 * df.isna().mean()).round(2),
})

print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
print(f"Embedded Stata value-label maps: {len(meta.value_labels or {})}")
display(var_table.head(40))
display(
    var_table.sort_values("missing_pct", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
df.info(max_cols=20)

Variables: 1,093  |  Observations: 928
Embedded Stata value-label maps: 0


,column,stata_label,dtype,missing_n,missing_pct
id,id,UNIQUE IDENTIFICATION,str,0,0.00
Q402,Q402,"402. The first time you had sex, was it becaus...",object,454,48.92
Q402B,Q402B,"402B. The first time you had sex, were you phy...",object,900,96.98
psu,psu,EA ID,int64,0,0.00
hh,hh,household,int64,0,0.00
prov,prov,province,int64,0,0.00
dist,dist,district,int64,0,0.00
const,const,const,int64,0,0.00
ward,ward,ward,int64,0,0.00
csa,csa,csa,int64,0,0.00


,column,stata_label,dtype,missing_n,missing_pct
0,ERDAY_23,DATE OF BIRTH OF ER -DAY(23),object,928,100.0
1,YOB_24,Date of Birth -Year(24),object,928,100.0
2,ERDAY_17,DATE OF BIRTH OF ER -DAY(17),object,928,100.0
3,ERDAY_16,DATE OF BIRTH OF ER -DAY(16),object,928,100.0
4,ERDAY_15,DATE OF BIRTH OF ER -DAY(15),object,928,100.0
5,ERDAY_14,DATE OF BIRTH OF ER -DAY(14),object,928,100.0
6,ERDAY_13,DATE OF BIRTH OF ER -DAY(13),object,928,100.0
7,ERDAY_12,DATE OF BIRTH OF ER -DAY(12),object,928,100.0
8,ERDAY_11,DATE OF BIRTH OF ER -DAY(11),object,928,100.0
9,ERDAY_10,DATE OF BIRTH OF ER -DAY(10),object,928,100.0


<class 'pandas.DataFrame'>
RangeIndex: 928 entries, 0 to 927
Columns: 1093 entries, id to Finalwgt
dtypes: float64(1), int64(120), object(701), str(271)
memory usage: 7.7+ MB


### Harmonized geography / ID checklist (`utils.checklist`)

**Admin 2 = enumeration areas (EAs):** Stata labels **`psu`** as *EA ID*—use **`psu`** for **GeoLevel2 / Admin 2** (not **`dist`**, which is the **district** code layer). **`psu`** is also the **cluster** for design—duplicate Excel rows or cross-reference in **notes** (`skills/memory.md`).

This cell summarizes **`df`** (Male PUD loaded above). Female PUD uses the **same** Stata names for these fields—re-run after loading female if needed.

Edit **`CANDIDATES`** after §2 if you change the column set.


In [5]:
# You choose candidates after §2 EDA; utils summarize columns present in `df`.
from utils.checklist import build_checklist_df, checklist_to_tsv

CANDIDATES = [
    ("Admin 1 (province)", ["prov"]),
    ("District code (~1.5)", ["dist"]),
    ("Admin 2 — enumeration area (EA)", ["psu"]),
    ("Cluster (svy; same EA as psu)", ["psu"]),
    ("Household # (within EA)", ["hh"]),
    ("Respondent ID", ["id"]),
    ("Finer geo / optional", ["ward", "const", "csa"]),
    ("Roster / selection context", ["nsel", "ntot"]),
    ("Split-sample / design", ["hcluster"]),
    ("Field / visit date parts", ["HYR_VF", "HMTH_VF", "HDAY_VF", "VISIT_NF"]),
]

_labels = meta.column_names_to_labels or {}
checklist_df = build_checklist_df(df, CANDIDATES, column_labels=_labels)

with pd.option_context("display.max_colwidth", 100, "display.width", 220):
    display(checklist_df)

print("\n--- TSV (copy for Excel / codebook) ---\n")
print(checklist_to_tsv(checklist_df))


,slot,column,stata_label,type_and_width,suggested_layout,dtype,nunique,missing_n,missing_pct,pi_digits_char_usual_display,min_nonnull,max_nonnull,sample_first_3,slot_notes
0,Admin 1 (province),prov,province,int; 1–2 digits,<NA>,int64,10,0,0.0,1–2,1,10,"1, 1, 1",<NA>
1,District code (~1.5),dist,district,int; 3–4 digits,<NA>,int64,61,0,0.0,3–4,101,1007,"101, 101, 101",<NA>
2,Admin 2 — enumeration area (EA),psu,EA ID,int; 1–3 digits,<NA>,int64,133,0,0.0,1–3,1,241,"1, 1, 1",<NA>
3,Cluster (svy; same EA as psu),psu,EA ID,int; 1–3 digits,<NA>,int64,133,0,0.0,1–3,1,241,"1, 1, 1",<NA>
4,Household # (within EA),hh,household,int; 1–3 digits,<NA>,int64,160,0,0.0,1–3,1,222,"9, 20, 24",<NA>
5,Respondent ID,id,UNIQUE IDENTIFICATION,str; 6 digits,<NA>,str,928,0,0.0,6,<NA>,<NA>,"'001009', '001020', '001024'",<NA>
6,Finer geo / optional,ward,ward,int; 1–2 digits,<NA>,int64,28,0,0.0,1–2,1,33,"2, 2, 2",<NA>
7,Finer geo / optional,const,const,int; 1–3 digits,<NA>,int64,95,0,0.0,1–3,1,147,"1, 1, 1",<NA>
8,Finer geo / optional,csa,csa,int; 1–2 digits,<NA>,int64,21,0,0.0,1–2,1,31,"1, 1, 1",<NA>
9,Roster / selection context,nsel,Number of females in this household,int; 1 digit,<NA>,int64,5,0,0.0,1,1,5,"2, 1, 2",<NA>



--- TSV (copy for Excel / codebook) ---

slot	column	stata_label	type_and_width	suggested_layout	dtype	nunique	missing_n	missing_pct	pi_digits_char_usual_display	min_nonnull	max_nonnull	sample_first_3	slot_notes
Admin 1 (province)	prov	province	int; 1–2 digits		int64	10	0	0.0	1–2	1	10	1, 1, 1	
District code (~1.5)	dist	district	int; 3–4 digits		int64	61	0	0.0	3–4	101	1007	101, 101, 101	
Admin 2 — enumeration area (EA)	psu	EA ID	int; 1–3 digits		int64	133	0	0.0	1–3	1	241	1, 1, 1	
Cluster (svy; same EA as psu)	psu	EA ID	int; 1–3 digits		int64	133	0	0.0	1–3	1	241	1, 1, 1	
Household # (within EA)	hh	household	int; 1–3 digits		int64	160	0	0.0	1–3	1	222	9, 20, 24	
Respondent ID	id	UNIQUE IDENTIFICATION	str; 6 digits		str	928	0	0.0	6			'001009', '001020', '001024'	
Finer geo / optional	ward	ward	int; 1–2 digits		int64	28	0	0.0	1–2	1	33	2, 2, 2	
Finer geo / optional	const	const	int; 1–3 digits		int64	95	0	0.0	1–3	1	147	1, 1, 1	
Finer geo / optional	csa	csa	int; 1–2 digits		int64	21	0	0.0	1–2	